# Task 3: Forecasting Models — Training

This notebook trains three models on each of the three target areas:
- **SARIMA** — Classical statistical model with seasonal support
- **LSTM** — Recurrent neural network with gated memory
- **Transformer** — Self-attention-based sequence model

**Test period:** December 16–22 (strictly held out — never used for training or validation)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import torch
import pickle

from src.data.loader import (
    load_area_from_parquet, get_area_series, total_traffic_per_area,
)
from src.data.preprocessor import (
    train_test_split_by_date, fit_scaler, scale_series,
    inverse_scale, create_sequences, fill_missing,
)
from src.models.sarima_model import run_sarima
from src.models.lstm_model import run_lstm
from src.models.transformer_model import run_transformer
from src.utils.timer import get_hardware_info

PARQUET_DIR = '../data/processed/milan_traffic_parquet/'
TOTALS_PATH = '../data/processed/total_per_area.parquet'
MODELS_DIR  = '../outputs/models/'
os.makedirs(MODELS_DIR, exist_ok=True)

SEQ_LENGTH = 144  # 1 day = 144 x 10-min intervals

print('Hardware info:')
for k, v in get_hardware_info().items():
    print(f'  {k}: {v}')

Hardware info:
  platform: Windows-10-10.0.22631-SP0
  processor: Intel64 Family 6 Model 186 Stepping 3, GenuineIntel
  python_version: 3.11.9
  cuda_available: False


## 3.1 Identify Target Areas

In [2]:
if os.path.exists(TOTALS_PATH):
    totals = pd.read_parquet(TOTALS_PATH).iloc[:, 0]
    totals.index = totals.index.astype('int32')
else:
    totals = total_traffic_per_area(PARQUET_DIR)
    totals.to_frame().to_parquet(TOTALS_PATH)

top_area = int(totals.idxmax())
TARGET_AREAS = {
    f'Area_{top_area}_TopTraffic': top_area,
    'Area_4159': 4159,
    'Area_4556': 4556,
}
print('Target areas:', TARGET_AREAS)

Target areas: {'Area_5161_TopTraffic': 5161, 'Area_4159': 4159, 'Area_4556': 4556}


## 3.2 Preprocessing Pipeline

For each area:
1. Extract time series
2. Fill missing timesteps via linear interpolation
3. Split: train = all data before Dec 16, test = Dec 16–22
4. Fit MinMaxScaler on train only, apply to test
5. Build overlapping sequences of length 144 for neural models

In [3]:
# Selectively read the 3 target areas (predicate pushdown — no full-dataset load)
df_3 = load_area_from_parquet(PARQUET_DIR, list(TARGET_AREAS.values()))
df_3['datetime'] = pd.to_datetime(df_3['time_interval'], unit='ms')
print(f'Loaded {len(df_3):,} rows for {df_3["square_id"].nunique()} areas')

area_data = {}
for label, sq_id in TARGET_AREAS.items():
    series = get_area_series(df_3, sq_id)
    series = fill_missing(series)

    train, test = train_test_split_by_date(series)
    train_scaled, scaler = fit_scaler(train)
    test_scaled = scale_series(test, scaler)

    # Sequences for neural models — built over the concatenated stream so the
    # first test sample's context is the tail of the train window.
    full_scaled = np.concatenate([train_scaled, test_scaled])
    X_all, y_all = create_sequences(full_scaled, SEQ_LENGTH)

    n_train = len(train_scaled) - SEQ_LENGTH
    X_train, y_train = X_all[:n_train], y_all[:n_train]
    X_test,  y_test  = X_all[n_train:], y_all[n_train:]
    y_test_orig = inverse_scale(y_test, scaler)

    area_data[label] = {
        'train': train, 'test': test, 'scaler': scaler,
        'X_train': X_train, 'y_train': y_train,
        'X_test':  X_test,  'y_test_orig': y_test_orig,
    }
    print(f'{label}: train={len(train)}, test={len(test)}, X_train={X_train.shape}')

Loaded 26,784 rows for 3 areas
Area_5161_TopTraffic: train=6486, test=2442, X_train=(6342, 144)
Area_4159: train=6486, test=2442, X_train=(6342, 144)
Area_4556: train=6486, test=2442, X_train=(6342, 144)


## 3.2b Baselines: persistence & seasonal-naive

Anchor the comparison with two zero-learning baselines so the trained
models can be judged against a non-trivial floor. Persistence
($\\hat{y}_{t+1}=y_t$) is what a SARIMA without seasonality would degrade
to; seasonal-naive ($\\hat{y}_{t+1}=y_{t+1-144}$) embeds the strong daily
cycle that ACF Lag-144 in Task 2 already exposed.

In [4]:
from src.models.baseline import run_persistence, run_seasonal_naive

persistence_results    = {}
seasonal_naive_results = {}
for label, data in area_data.items():
    print(f'\n--- {label} ---')
    persistence_results[label]    = run_persistence(data['train'], data['test'])
    seasonal_naive_results[label] = run_seasonal_naive(
        data['train'], data['test'], period=144
    )


--- Area_5161_TopTraffic ---
Persistence -> MAE: 78.8715 | MAPE: 11.06% | RMSE: 119.7306
SeasonalNaive(s=144) -> MAE: 398.7463 | MAPE: 57.28% | RMSE: 690.1423

--- Area_4159 ---
Persistence -> MAE: 12.5299 | MAPE: 8.51% | RMSE: 17.1979
SeasonalNaive(s=144) -> MAE: 39.2428 | MAPE: 25.39% | RMSE: 63.1034

--- Area_4556 ---
Persistence -> MAE: 25.4983 | MAPE: 7.99% | RMSE: 35.3194
SeasonalNaive(s=144) -> MAE: 63.5850 | MAPE: 20.15% | RMSE: 87.6842


## 3.2c Hyperparameter selection

**SARIMA.** AIC grid over a small (p,d,q)(P,D,Q,144) cube, fit on the last
7 days of the training set (subsampled to keep search time manageable —
each fit at s=144 already costs tens of seconds). Best order is re-fit on
the full training series in section 3.3.

**LSTM / Transformer.** Small random search on a validation tail of the
training series for *one* representative area (the top-traffic cell); the
chosen config is then reused across all three areas to keep total compute
tractable. This is documented as a deliberate trade-off.

In [5]:
from src.models.tuning import sarima_aic_search

# Tune on the top-traffic area (cleanest signal); reuse order for the others.
ref_label = next(iter(area_data))
sarima_grid = sarima_aic_search(
    area_data[ref_label]['train'],
    p_range=(0, 1, 2), d_range=(0, 1), q_range=(0, 1, 2),
    P_range=(0, 1), D_range=(0, 1), Q_range=(0, 1),
    s=144, max_combos=18, subsample=144*7,
)
print(sarima_grid.head(10).to_string())
best_sarima_order    = sarima_grid.iloc[0]['order']
best_sarima_seasonal = sarima_grid.iloc[0]['seasonal_order']
print(f'\nSelected: order={best_sarima_order}, seasonal_order={best_sarima_seasonal}')

       order  seasonal_order           aic           bic  fit_seconds  converged
0  (2, 1, 2)  (0, 0, 0, 144)  13050.070022  13074.628758         0.20       True
1  (1, 1, 2)  (0, 0, 0, 144)  13051.397674  13071.044664         0.16       True
2  (2, 0, 2)  (0, 0, 0, 144)  13059.542429  13084.106143         0.17       True
3  (2, 1, 1)  (0, 0, 0, 144)  13097.949191  13117.600162         0.19       True
4  (1, 1, 1)  (0, 0, 0, 144)  13178.292391  13193.030619         0.12       True
5  (0, 1, 2)  (0, 0, 0, 144)  13204.588050  13219.323292         0.04       True
6  (1, 0, 2)  (0, 0, 0, 144)  13217.412666  13237.063637         0.06       True
7  (2, 1, 0)  (0, 0, 0, 144)  13227.768753  13242.506981         0.01       True
8  (0, 1, 1)  (0, 0, 0, 144)  13234.308800  13244.134285         0.03       True
9  (1, 1, 0)  (0, 0, 0, 144)  13244.800582  13254.628057         0.02       True

Selected: order=(2, 1, 2), seasonal_order=(0, 0, 0, 144)


In [6]:
from src.models.tuning import (
    random_search_neural, LSTM_SEARCH_SPACE, TRANSFORMER_SEARCH_SPACE,
)
from src.models.lstm_model import train_lstm, DEFAULT_CONFIG as LSTM_BASE
from src.models.transformer_model import train_transformer, DEFAULT_CONFIG as TRANS_BASE

# Use a smaller epoch budget for search; final training uses full epochs.
lstm_search_cfg  = {**LSTM_BASE,  'epochs': 8}
trans_search_cfg = {**TRANS_BASE, 'epochs': 8}

ref = area_data[ref_label]
print('LSTM random search:')
lstm_results_df = random_search_neural(
    train_lstm, ref['X_train'], ref['y_train'],
    base_config=lstm_search_cfg, search_space=LSTM_SEARCH_SPACE,
    n_trials=6, val_frac=0.1, seed=0,
)
best_lstm_cfg = {**LSTM_BASE, **lstm_results_df.iloc[0]['config'], 'epochs': 30}
print(f"\nLSTM chosen: {lstm_results_df.iloc[0]['config']}")

print('\nTransformer random search:')
trans_results_df = random_search_neural(
    train_transformer, ref['X_train'], ref['y_train'],
    base_config=trans_search_cfg, search_space=TRANSFORMER_SEARCH_SPACE,
    n_trials=6, val_frac=0.1, seed=0,
)
best_trans_cfg = {**TRANS_BASE, **trans_results_df.iloc[0]['config'], 'epochs': 30}
print(f"\nTransformer chosen: {trans_results_df.iloc[0]['config']}")

LSTM random search:
Training LSTM on: cpu


  Epoch 5/8 — Train Loss: 0.001405 | Val Loss: 0.001164


  trial 0: val_mse=0.000769  cfg={'hidden_size': 128, 'num_layers': 2, 'dropout': 0.1, 'lr': 0.0005}
Training LSTM on: cpu


  Epoch 5/8 — Train Loss: 0.001318 | Val Loss: 0.000763


  trial 1: val_mse=0.000658  cfg={'hidden_size': 256, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.0005}
Training LSTM on: cpu


...

  trial 3: val_mse=0.002113  cfg={'d_model': 128, 'nhead': 2, 'num_layers': 2, 'dim_feedforward': 128, 'dropout': 0.1, 'lr': 0.0002}
Training Transformer on: cpu


  Epoch 5/8 — Train Loss: 0.002501 | Val Loss: 0.005517


  trial 4: val_mse=0.004255  cfg={'d_model': 64, 'nhead': 4, 'num_layers': 3, 'dim_feedforward': 128, 'dropout': 0.2, 'lr': 0.0005}
Training Transformer on: cpu


  Epoch 5/8 — Train Loss: 0.003207 | Val Loss: 0.003584


  trial 5: val_mse=0.002884  cfg={'d_model': 64, 'nhead': 2, 'num_layers': 3, 'dim_feedforward': 256, 'dropout': 0.2, 'lr': 0.0002}

Transformer chosen: {'d_model': 64, 'nhead': 4, 'num_layers': 2, 'dim_feedforward': 512, 'dropout': 0.1, 'lr': 0.0002}


## 3.3 SARIMA Training

SARIMA(1,1,1)(1,1,1,144) — daily seasonality (s=144 = 1 day × 144 intervals of 10min).

> Note: SARIMA is computationally expensive at s=144. Consider reducing to s=48 (8 hours) if runtime is too high, or using a 1-week sample for hyperparameter selection.

In [7]:
sarima_results = {}

for label, data in area_data.items():
    print(f'\n{"="*50}')
    print(f'SARIMA — {label}')
    result = run_sarima(
        data['train'], data['test'],
        order=best_sarima_order, seasonal_order=best_sarima_seasonal,
    )
    sarima_results[label] = result
    print(f"Timer: train={result['timer'].train_time:.1f}s | "
          f"inference={result['timer'].inference_time:.4f}s")


SARIMA — Area_5161_TopTraffic


SARIMA training complete. AIC: 88671.79


                               SARIMAX Results                                
Dep. Variable:              area_5161   No. Observations:                 6486
Model:               SARIMAX(2, 1, 2)   Log Likelihood              -44330.897
Date:                Mon, 18 May 2026   AIC                          88671.795
Time:                        18:05:22   BIC                          88705.679
Sample:                    10-31-2013   HQIC                         88683.516
                         - 12-15-2013                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          1.0957      0.016     69.748      0.000       1.065       1.127
ar.L2         -0.1428      0.016     -9.125      0.000      -0.173      -0.112
ma.L1         -1.5008      0.015   -103.368      0.0


SARIMA Metrics → MAE: 74.9121 | MAPE: 10.39% | RMSE: 115.4374
Timer: train=1.9s | inference=101.4562s

SARIMA — Area_4159


SARIMA training complete. AIC: 67018.38
                               SARIMAX Results                                
Dep. Variable:              area_4159   No. Observations:                 6486
Model:               SARIMAX(2, 1, 2)   Log Likelihood              -33504.188
Date:                Mon, 18 May 2026   AIC                          67018.375
Time:                        18:07:08   BIC                          67052.259
Sample:                    10-31-2013   HQIC                         67030.096
                         - 12-15-2013                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.4688      0.134     -3.486      0.000      -0.732      -0.205
ar.L2         -0.0976      0.026     -3.826      0.000      -0.148      -0.048
ma.L1       


SARIMA Metrics → MAE: 11.4571 | MAPE: 7.63% | RMSE: 16.0795
Timer: train=0.5s | inference=87.8769s

SARIMA — Area_4556


SARIMA training complete. AIC: 74080.16
                               SARIMAX Results                                
Dep. Variable:              area_4556   No. Observations:                 6486
Model:               SARIMAX(2, 1, 2)   Log Likelihood              -37035.080
Date:                Mon, 18 May 2026   AIC                          74080.161
Time:                        18:08:38   BIC                          74114.045
Sample:                    10-31-2013   HQIC                         74091.882
                         - 12-15-2013                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          1.0786      0.018     59.285      0.000       1.043       1.114
ar.L2         -0.1738      0.017    -10.016      0.000      -0.208      -0.140
ma.L1       


SARIMA Metrics → MAE: 23.5145 | MAPE: 7.28% | RMSE: 32.6205
Timer: train=1.5s | inference=85.4561s


## 3.4 LSTM Training

In [8]:
lstm_results = {}
for label, data in area_data.items():
    print(f'\n{"="*50}')
    print(f'LSTM — {label}')
    result = run_lstm(
        data['X_train'], data['y_train'],
        data['X_test'], data['y_test_orig'],
        data['scaler'], best_lstm_cfg,
    )
    lstm_results[label] = result
    torch.save(result, MODELS_DIR+f'lstm_{label}.pt')


LSTM — Area_5161_TopTraffic
Training LSTM on: cpu


  Epoch 5/30 — Train Loss: 0.001388 | Val Loss: 0.000820


  Epoch 10/30 — Train Loss: 0.001135 | Val Loss: 0.000745


  Epoch 15/30 — Train Loss: 0.001074 | Val Loss: 0.000575


  Epoch 20/30 — Train Loss: 0.001096 | Val Loss: 0.000497


...

  Epoch 15/30 — Train Loss: 0.002222 | Val Loss: 0.000722


  Epoch 20/30 — Train Loss: 0.002142 | Val Loss: 0.000805


  Epoch 25/30 — Train Loss: 0.002197 | Val Loss: 0.000670


  Epoch 30/30 — Train Loss: 0.002106 | Val Loss: 0.000694



LSTM Metrics → MAE: 24.3432 | MAPE: 7.72% | RMSE: 33.2231


## 3.5 Transformer Training

In [9]:
transformer_results = {}
for label, data in area_data.items():
    print(f'\n{"="*50}')
    print(f'Transformer — {label}')
    result = run_transformer(
        data['X_train'], data['y_train'],
        data['X_test'], data['y_test_orig'],
        data['scaler'], best_trans_cfg,
    )
    transformer_results[label] = result
    torch.save(result, MODELS_DIR+f'transformer_{label}.pt')


Transformer — Area_5161_TopTraffic
Training Transformer on: cpu


  Epoch 5/30 — Train Loss: 0.002080 | Val Loss: 0.002111


  Epoch 10/30 — Train Loss: 0.001567 | Val Loss: 0.002039


  Epoch 15/30 — Train Loss: 0.001570 | Val Loss: 0.002018


  Epoch 20/30 — Train Loss: 0.001622 | Val Loss: 0.002035


...

  Epoch 15/30 — Train Loss: 0.002321 | Val Loss: 0.000932


  Epoch 20/30 — Train Loss: 0.002275 | Val Loss: 0.000939


  Epoch 25/30 — Train Loss: 0.002289 | Val Loss: 0.000940


  Epoch 30/30 — Train Loss: 0.002282 | Val Loss: 0.000940



Transformer Metrics → MAE: 28.1157 | MAPE: 10.15% | RMSE: 36.4730


## 3.6 Save All Results

In [10]:
import pickle

all_results = {
    'persistence':    persistence_results,
    'seasonal_naive': seasonal_naive_results,
    'sarima':         sarima_results,
    'lstm':           lstm_results,
    'transformer':    transformer_results,
    'area_data':      {k: {'test': v['test'], 'y_test_orig': v['y_test_orig']}
                       for k, v in area_data.items()},
    'tuning': {
        'sarima_grid':        sarima_grid,
        'lstm_search':        lstm_results_df,
        'transformer_search': trans_results_df,
        'best_sarima':        {'order': best_sarima_order,
                               'seasonal_order': best_sarima_seasonal},
        'best_lstm':          best_lstm_cfg,
        'best_transformer':   best_trans_cfg,
    },
}

with open(MODELS_DIR+'all_results.pkl', 'wb') as f:
    pickle.dump(all_results, f)
print('All results saved.')

All results saved.
